# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/singhmahip688-hue/flyrank-ml-internhip/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install -q duckdb huggingface_hub

In [2]:
import duckdb

con = duckdb.connect()

In [3]:
from google.colab import userdata
import duckdb

con = duckdb.connect()
con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{userdata.get("HF_TOKEN")}'
)
""")

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one row = one (client, content item, report_date) —
i.e. one day's performance for one piece of content belonging to one client.
This is the grain of fact_content_daily_performance.

Time window: the full warehouse spans 2025-01-27 to 2026-06-30 (~17 months),
but client history depth is unbalanced — I will check each client's
gsc_data_start / ga4_data_start before trusting any date range.

For modeling, I will develop on a mid-panel month (e.g. month=2026-03) and
treat the final month (June 2026) as a sealed test window — never touched
during feature/label design, since fact_content_daily_performance_sample
IS that final month and using it early would mean developing inside my
own test set.

In [4]:
import duckdb
print(duckdb.__version__)

1.3.2


In [5]:
import huggingface_hub
print("huggingface_hub version:", huggingface_hub.__version__)

huggingface_hub version: 1.23.0


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Grain + row count + date range check on the daily fact table
q = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
"""

print(con.execute(q).fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count start_date   end_date
0   78835655 2025-01-27 2026-06-30


In [7]:
tables = {
    "dim_clients": "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet",
    "dim_content": "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet",
    "fact_content_daily_performance": "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet",
    "fact_content_query_90d": "hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet"
}

for name, path in tables.items():
    print("\n" + "="*60)
    print(name)
    print("="*60)

    q = f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{path}')
    """

    print(con.execute(q).fetchdf())


dim_clients
           column_name column_type null   key default extra
0       client_hash_id     VARCHAR  YES  None    None  None
1            is_active     BOOLEAN  YES  None    None  None
2       has_gsc_access     BOOLEAN  YES  None    None  None
3       has_ga4_access     BOOLEAN  YES  None    None  None
4       access_profile     VARCHAR  YES  None    None  None
5  client_created_date        DATE  YES  None    None  None
6  client_updated_date        DATE  YES  None    None  None
7       gsc_data_start        DATE  YES  None    None  None
8       ga4_data_start        DATE  YES  None    None  None

dim_content
                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count    

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: gsc_avg_position, impressions, content_type, prior-window CTR/engagement
(from *_prev30-style columns only — never same-window query features, to avoid
window overlap leakage), ga4_data_available flag.

Label / proxy: CTR opportunity score = expected_ctr(position bucket) − actual_ctr.
A positive gap = under-clicked despite good visibility. This is a scoring/ranking
target, not a classifier — matches the CTR/Engagement Opportunity Scoring lane.

Context: client_hash_id, content_id, report_date — used only for joining, grouping,
and client-holdout splitting. Never fed to the model as a feature.

Excluded:
- trend_direction / trend_pct — these are what is_declining_label is derived from;
  using them anywhere near CTR features risks circularity/leakage.
- raw query text (from fact_content_query_90d) — excluded, privacy risk,
  fine only as ANY_VALUE() aggregated context, never as a per-row feature.
- GA4 columns on rows before a client's ga4_data_start — these are zero-FILLED,
  not real zeros. Must filter using ga4_data_available, not just check for null.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q = """
DESCRIBE
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
LIMIT 1
"""

print(con.execute(q).fetchdf())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Claim 1 (grain holds — no duplicate client/content/date rows): verified below.
Claim 2 (row count matches known total): verified above in Section 1.
Claim 3 (GA4 zero-fill pattern is real, not missing-at-random): verified below.
Claim 4 (client history is unbalanced — gsc_data_start varies): verified below.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q = """
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS c
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY 1,2,3
HAVING c > 1
"""
dupes = con.execute(q).fetchdf()
print(dupes)
print("Duplicate (client, content, date) combos in March:", len(dupes))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, c]
Index: []
Duplicate (client, content, date) combos in March: 0


In [10]:
q = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""
print(con.execute(q).fetchdf())

   row_count start_date   end_date
0    9841378 2026-03-01 2026-03-31


In [11]:
q = """
SELECT COUNT(*) AS total_rows_march
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""
total_march = con.execute(q).fetchdf()
print(total_march)

q = """
SELECT COUNT(*) AS rows_with_ga4_available
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND ga4_data_available IS TRUE
"""
available_march = con.execute(q).fetchdf()
print(available_march)

   total_rows_march
0           9841378


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rows_with_ga4_available
0                   413966


## Five features + the leakage trap

In [12]:
import pandas as pd
q = """
SELECT
    client_hash_id, content_hash_id, report_date,
    gsc_impressions, gsc_clicks, gsc_avg_position,
    ga4_total_engagement_sec, ga4_data_available
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND ga4_data_available IS TRUE
  AND gsc_impressions >= 100
"""
df = con.execute(q).fetchdf()

df["gsc_ctr"] = df["gsc_clicks"] / df["gsc_impressions"]

df["position_tier"] = pd.cut(
    df["gsc_avg_position"], bins=[0, 3, 10, 20, 1000],
    labels=["top3", "top10", "top20", "beyond20"]
)
df["expected_ctr"] = df.groupby("position_tier")["gsc_ctr"].transform("mean")
df["ctr_gap"] = df["expected_ctr"] - df["gsc_ctr"]

df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_16831/3934531090.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["expected_ctr"] = df.groupby("position_tier")["gsc_ctr"].transform("mean")


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_total_engagement_sec,ga4_data_available,gsc_ctr,position_tier,expected_ctr,ctr_gap
0,client_65de48885f4ef01b,content_e25ea7297a1dffd3,2026-03-01,179,0,5.156425,0,True,0.000000,top10,0.006416,0.006416
1,client_65de48885f4ef01b,content_62673eea26c31c17,2026-03-01,3282,1,6.167885,0,True,0.000305,top10,0.006416,0.006112
2,client_c182d11e4862a37d,content_f4a0e5c90b283626,2026-03-01,250,0,19.448000,0,True,0.000000,top20,0.005556,0.005556
3,client_c182d11e4862a37d,content_da76e1818babb4ce,2026-03-01,132,0,11.348485,0,True,0.000000,top20,0.005556,0.005556
4,client_c182d11e4862a37d,content_07391c5144deb0b6,2026-03-01,152,1,7.401316,28,True,0.006579,top10,0.006416,-0.000162
5,client_c182d11e4862a37d,content_cfb3278abccdb93c,2026-03-01,201,1,4.761194,0,True,0.004975,top10,0.006416,0.001441
6,client_a2eeb8899886adde,content_9defca86778485fe,2026-03-01,109,0,4.275229,0,True,0.000000,top10,0.006416,0.006416
7,client_e547b89c05043229,content_a6eb550e132505fd,2026-03-02,504,5,6.236111,0,True,0.009921,top10,0.006416,-0.003504
8,client_e547b89c05043229,content_cbb44b4a10088d7b,2026-03-02,192,0,2.619792,0,True,0.000000,top3,0.007136,0.007136
9,client_e547b89c05043229,content_bef2d5b965ab9c14,2026-03-02,130,1,6.792308,0,True,0.007692,top10,0.006416,-0.001276


Five features:

1. **gsc_impressions** — knowable because it's a completed measurement of
   March search visibility, already observed by the time we'd act on it.
2. **gsc_avg_position** — knowable because it's the already-observed average
   ranking position for the month, not a future value.
3. **position_tier** — knowable because it's a deterministic bucket computed
   directly from gsc_avg_position, which is itself already known.
4. **content_type** (would join from dim_content on content_hash_id) —
   knowable because it's a static, publish-time attribute of the page.
5. **ga4_total_engagement_sec** — knowable because it's a completed
   engagement measurement from the same historical window.

Excluded: gsc_ctr and expected_ctr are the ingredients used to compute
ctr_gap itself, so they can never be used as model inputs — that's exactly
the trap demonstrated below.

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

honest_features = ["gsc_impressions", "gsc_avg_position", "ga4_total_engagement_sec"]
df_clean = df.dropna(subset=honest_features + ["ctr_gap"])

X, y = df_clean[honest_features], df_clean["ctr_gap"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression().fit(X_train, y_train)
honest_r2 = r2_score(y_test, model.predict(X_test))
print("Honest R²:", honest_r2)

# THE TRAP: add the label's own ingredient back in as a "feature"
leaky_features = honest_features + ["gsc_ctr"]
X_leak = df_clean[leaky_features]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.2, random_state=42)
leaky_model = LinearRegression().fit(X_train_l, y_train_l)
leaky_r2 = r2_score(y_test_l, leaky_model.predict(X_test_l))
print("Leaky R²:", leaky_r2)

print(f"\nJump: {honest_r2:.3f} -> {leaky_r2:.3f}")
print("gsc_ctr is removed from the final feature set. Keeping only the honest R².")

Honest R²: 0.0212997888651133
Leaky R²: 0.9850757029729906

Jump: 0.021 -> 0.985
gsc_ctr is removed from the final feature set. Keeping only the honest R².


The leak: gsc_ctr is one of the two direct ingredients used to compute
ctr_gap (expected_ctr - gsc_ctr). Feeding it back in as a "feature" lets
the model just reverse the arithmetic instead of learning anything real —
hence the score jumping toward perfect. It has been removed; the honest
R² above is the number I'm keeping.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data cannot tell us WHY a page is under-clicked — only that the gap exists.
History is unbalanced across clients — gsc_data_start / ga4_data_start differ,
so a global date window would silently exclude or misrepresent newer clients.
Some clients have little or no usable search/analytics history and must
be filtered, not imputed.
fact_content_query_90d has overlapping windows with the daily fact table's final
months — query-level features must be window-aligned before joining, or they leak
future information into the label.
The final month is treated as a sealed test set — no label or feature logic
may be developed by looking at it.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q = """
SELECT COUNT(*) AS thin_history_clients
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
WHERE gsc_data_start > '2026-04-01'
"""
print(con.execute(q).fetchdf())

   thin_history_clients
0                    10


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.